In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.3 LoRA

> 🎯 **Goal:** Fine-tune a model by training a tiny set of bolt-on 'adjustment
> knobs' while the original weights stay frozen.

**Why this matters.** Full fine-tuning (the SFT in l7.2) updates *every* weight,
which means a full optimizer state and a fresh copy of the whole model per task.
For large models that is expensive in memory and storage. LoRA (Low-Rank
Adaptation) makes adapting a model cheap enough to do on modest hardware, and lets
you keep dozens of small task-specific adapters instead of dozens of full models.

**The intuition.** Instead of repainting the whole house to change one room, you
bolt a few small adjustment knobs onto the existing walls and turn only those. The
house (the **frozen** base model) is untouched and shared, the knobs (the
**adapter**) are tiny and specific to your task. Want a different task tomorrow?
Unbolt this set of knobs, bolt on another. The expensive base never changes.

**The mechanics.** Some key terms. **Frozen weights** are parameters with gradients
turned off: they participate in the forward pass but never update. An **adapter** is
the small set of *new* trainable parameters you add. LoRA's specific adapter is a
**low-rank update**: for a weight matrix it learns two skinny matrices `A` and `B`
and adds their product `B @ A` to the frozen weight. Because `A` and `B` have a
small inner dimension (the **rank**, here 4), their product has far fewer free
parameters than the full matrix, often well under a few percent of the model. We
freeze the base, add LoRA, and train only the adapter, then check how few parameters
actually move.

In [ ]:
from pocketlm import (train_tiny, pick_config, add_lora, trainable_parameters,
                      make_batch)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
total = sum(p.numel() for p in model.parameters())
add_lora(model, rank=4)
trainable = sum(p.numel() for p in trainable_parameters(model))
print(f"trainable: {trainable} / {total}  ({100 * trainable / total:.1f}%)")

data = torch.tensor(tok.encode(corpus), dtype=torch.long)
opt = torch.optim.AdamW(trainable_parameters(model), lr=1e-3)
g = torch.Generator().manual_seed(0)
first = None
for _ in range(60 if os.environ.get("POCKETLM_CI") else 150):
    x, y = make_batch(data, model.cfg.block_size, 16, generator=g)
    _, loss = model(x, y)
    if first is None:
        first = loss.item()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("adapter-only loss:", round(first, 3), "->", round(loss.item(), 3))

**What just happened.** The first print showed the trainable count as a small
fraction of the total, often a couple of percent, because only the LoRA matrices
carry gradients. The second print showed the loss still moving down, proving the
tiny adapter is enough to steer the frozen base toward the new data. That is the
whole LoRA promise: most of the model is shared and frozen, only a sliver trains,
and you still get real adaptation.

⚠️ **Common pitfalls**
- Passing all parameters to the optimizer. You must hand it only
  `trainable_parameters(model)`, otherwise you are back to full fine-tuning and the
  frozen base is no longer frozen.
- Setting the rank too low or too high. Too low and the adapter cannot express the
  change, too high and you lose the memory savings. Rank is a knob to tune.
- Expecting LoRA to fix a base that lacks the underlying capability. Adapters nudge
  behavior, they do not add knowledge the base never learned.

🏋️ **Try it yourself.** Change `rank=4` to `rank=1` and then `rank=16`, rerun, and
watch the trainable-parameter percentage and the final loss move in opposite
directions. You are feeling the capacity-versus-cost trade that every LoRA setup
balances.

In [ ]:
assert 0 < trainable < total * 0.2
assert loss.item() <= first